# Adding a New Dataset

## CircuitBench Tutorial 8

This notebook demonstrates how to integrate a new benchmark dataset into CircuitBench while maintaining reproducibility, metadata integrity, and compatibility with benchmarking workflows.

### Learning Objectives

- Understand the CircuitBench dataset structure
- Validate dataset integrity
- Register a new benchmark dataset
- Generate dataset metadata
- Perform automated quality checks
- Export a reusable benchmark package


## Scientific Background

A benchmark dataset should be reproducible, well documented, machine-readable, and accompanied by complete metadata. CircuitBench standardizes dataset registration to ensure consistency across benchmarks.

In [ ]:
from pathlib import Path
import json
import warnings
import random

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from circuitbench.datasets import DatasetRegistry

## Initialize Dataset Registry

In [ ]:
registry = DatasetRegistry()

## Existing Registered Datasets

In [ ]:
registry.list_datasets()

## Specify Dataset Location

In [ ]:
dataset_path = Path("datasets/My_New_Dataset.csv")
dataset_path

## Load Dataset

In [ ]:
dataset = pd.read_csv(dataset_path)
dataset.head()

## Dataset Overview

In [ ]:
print("Samples :", dataset.shape[0])
print("Features:", dataset.shape[1])

dataset.info()

## Validate Dataset Structure

In [ ]:
validation = {
    "Missing Values": dataset.isnull().sum().sum(),
    "Duplicate Rows": dataset.duplicated().sum(),
    "Number of Features": dataset.shape[1],
    "Number of Samples": dataset.shape[0],
}

pd.Series(validation)

## Dataset Summary Statistics

In [ ]:
dataset.describe(include="all")

## Define Target Variable

In [ ]:
target_column = "target"

X = dataset.drop(columns=[target_column])
y = dataset[target_column]

print("Feature matrix:", X.shape)
print("Target vector :", y.shape)

## Generate Dataset Metadata

In [ ]:
metadata = {
    "dataset_name": "My_New_Dataset",
    "samples": int(dataset.shape[0]),
    "features": int(dataset.shape[1] - 1),
    "target": target_column,
    "missing_values": int(dataset.isnull().sum().sum()),
    "duplicates": int(dataset.duplicated().sum()),
    "random_seed": SEED,
    "version": "1.0.0",
}

metadata

## Register Dataset

In [ ]:
registry.register(
    name="My_New_Dataset", dataframe=dataset, target=target_column, metadata=metadata
)

## Verify Registration

In [ ]:
registry.list_datasets()

## Export Metadata

In [ ]:
output_dir = Path("registered_dataset")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

## Train/Test Split Validation

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

print("Training Samples:", len(X_train))
print("Testing Samples :", len(X_test))

## Dataset Quality Checklist

In [ ]:
quality = pd.DataFrame(
    {
        "Check": [
            "Missing Values",
            "Duplicate Rows",
            "Target Column Exists",
            "Minimum Samples (>100)",
            "Minimum Features (>2)",
        ],
        "Status": [
            dataset.isnull().sum().sum() == 0,
            dataset.duplicated().sum() == 0,
            target_column in dataset.columns,
            dataset.shape[0] > 100,
            dataset.shape[1] > 2,
        ],
    }
)

quality

## Feature Data Types

In [ ]:
feature_types = pd.DataFrame({"Feature": X.columns, "DataType": X.dtypes.astype(str)})

feature_types

## Preview Feature Correlation

In [ ]:
numeric_columns = X.select_dtypes(include=["number"])

correlation = numeric_columns.corr()

correlation.head()

## Save Dataset Package

In [ ]:
dataset.to_csv(output_dir / "dataset.csv", index=False)

feature_types.to_csv(output_dir / "feature_types.csv", index=False)

quality.to_csv(output_dir / "quality_report.csv", index=False)

## Registration Summary

In [ ]:
summary = {
    "dataset": metadata["dataset_name"],
    "samples": metadata["samples"],
    "features": metadata["features"],
    "registered": True,
    "output_directory": str(output_dir),
}

summary

## Export Registration Report

In [ ]:
with open(output_dir / "registration_report.json", "w") as f:
    json.dump(summary, f, indent=4)

## Reproducibility Checklist

- Dataset source documented.
- Metadata generated automatically.
- Missing values checked.
- Duplicate records verified.
- Feature data types recorded.
- Dataset registration validated.
- Registration report exported.
- Random seed documented.


## Best Practices

1. Preserve the original dataset in a read-only location.
2. Document all preprocessing performed before registration.
3. Include units, feature descriptions, and licensing information.
4. Validate every dataset before benchmarking.
5. Assign a version number whenever the dataset changes.
6. Store metadata alongside the dataset for reproducibility.


## Recommended Directory Structure

```
datasets/
├── My_New_Dataset/
│   ├── dataset.csv
│   ├── metadata.json
│   ├── README.md
│   ├── LICENSE
│   ├── CITATION.cff
│   ├── feature_types.csv
│   ├── quality_report.csv
│   └── registration_report.json
```


# Summary

In this notebook we:

- Loaded a new benchmark dataset.
- Validated structural integrity and data quality.
- Generated standardized metadata.
- Registered the dataset with CircuitBench.
- Produced quality assurance reports.
- Exported reusable dataset artifacts.

Following this workflow ensures that newly added datasets are reproducible, well documented, and immediately compatible with the CircuitBench benchmarking ecosystem.